## Import Libraries/Packages

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import torch

from datasets import load_dataset, Dataset
import wandb
import modal 
from modal import enter, exit, method, Image

from sentence_transformers.training_args import BatchSamplers

In [ ]:
load_dotenv(find_dotenv())
torch.cuda.is_available()

## Prepare Dataset

In [ ]:
train_dataset = load_dataset("potsu-potsu/mini-bioasq-with-metadata", name="question-answer-passages", split="train")
test_dataset = load_dataset("potsu-potsu/mini-bioasq-with-metadata", name="question-answer-passages", split="test")
corpus_dataset = load_dataset("potsu-potsu/mini-bioasq-with-metadata", name="text-corpus", split="test")

print(train_dataset[0], test_dataset[0], corpus_dataset[0], sep="\n")
print(len(train_dataset), len(test_dataset), len(corpus_dataset))

In [ ]:
#renaming columns to adhere to data structure and keys required by sentence transformer 
# https://sbert.net/docs/sentence_transformer/dataset_overview.html

train_dataset = train_dataset.rename_column("question", "anchor")
train_dataset = train_dataset.rename_column("answer", "positive")

test_dataset = test_dataset.rename_column("question", "anchor")
test_dataset = test_dataset.rename_column("answer", "positive")

corpus_dataset = corpus_dataset.rename_column("passage", "positive")

In [ ]:
# convert dataset into dictionary format required by the InformationRetrievalEvaluator
# corpus: maps corpus IDs to their text chunks (documents)
# format: {corpus_id: text_chunk}

corpus = dict(zip(corpus_dataset["id"], corpus_dataset["positive"]))
corpus

In [ ]:
# queries: maps query IDs to their questions
# format: {query_id: question_text}

queries = dict(zip(test_dataset["id"], test_dataset["anchor"]))
queries

In [ ]:
# queries: maps query IDs to the relevant documents
# Format: {query_id: [relevant cids]}

relevant_docs = dict(zip(test_dataset["id"], test_dataset["relevant_passage_ids"]))

In [ ]:
train_dict = {
    "anchor": [], 
    "positive": []
}

# multiple positive pairs per query based on relevant documents and human-annotated answer
for query, ans, rel_ids in zip(train_dataset["anchor"], train_dataset["positive"], train_dataset["relevant_passage_ids"]):
    train_dict["anchor"].extend([query]*(len(rel_ids)+1))
    train_dict["positive"].append(ans)
    for c_id in rel_ids:
        train_dict["positive"].append(corpus[c_id])

print(len(train_dict["anchor"]), len(train_dict["positive"]))

In [ ]:
assert len(train_dict["anchor"]) == len(train_dict["positive"])
train_dataset = Dataset.from_dict(train_dict)
train_dataset[0]

## Setup Modal Labs App

In [ ]:
app = modal.App("psi-bioasq")

In [ ]:
MODEL_DIR = "/model"
# BASE_MODEL = "NeuML/pubmedbert-base-embeddings"
# BASE_MODEL = "abhinand/MedEmbed-small-v0.1"
# BASE_MODEL = "abhinand/MedEmbed-base-v0.1"
# BASE_MODEL = "BAAI/bge-base-en-v1.5"
# BASE_MODEL = "nomic-ai/modernbert-embed-base"
# BASE_MODEL = "nomic-ai/nomic-embed-text-v1"
# BASE_MODEL = "Snowflake/snowflake-arctic-embed-m-long"
BASE_MODEL = "Snowflake/snowflake-arctic-embed-m-v1.5"


TIMEOUT = 3600

SECRETS = [modal.Secret.from_dict({
        "WANDB_PROJECT": os.environ["WANDB_PROJECT"],
        "WANDB_LOG_MODEL": os.environ["WANDB_LOG_MODEL"],
        "WANDB_API_KEY": os.environ["WANDB_API_KEY"], 
        "HUGGINGFACEHUB_API_TOKEN": os.environ["HUGGINGFACEHUB_API_TOKEN"], 
    })]

# GPU = "A10G"
GPU = "A100"

In [ ]:
matryoshka_dimensions = [768, 512, 256, 128, 64] # Important: large to small

query_prompt_name = None 
prompts = None 

if BASE_MODEL.startswith("Snowflake"):
    query_prompt_name = "query"

    prompts = {
        "anchor": "Represent this sentence for searching relevant passages: "
    }

print(query_prompt_name)
print(prompts)

In [ ]:
def convert_results_to_df(results, metrics: list[str] = None) -> pd.DataFrame:

    if metrics is None:
        metrics = [
            'ndcg@10',
            'mrr@10',
            'map@100',
            'accuracy@1',
            'accuracy@3',
            'accuracy@5',
            'accuracy@10',
            'precision@1',
            'precision@3',
            'precision@5',
            'precision@10',
            'recall@1',
            'recall@3',
            'recall@5',
            'recall@10'
        ]

    all_values = []

    # Print each metric
    for metric in metrics:
        values = []
        for dim in matryoshka_dimensions:
            key = f"dim_{dim}_cosine_{metric}"
            values.append(results[key])
        
        all_values.append(values)

    col_names = [str(dim) + "d" for dim in matryoshka_dimensions]
    df = pd.DataFrame(data=all_values, columns=col_names, index=metrics)
    df.loc["sequential score"] = ["-"]*(len(matryoshka_dimensions)-1) + [results['sequential_score']]
    
    return df

In [ ]:
def download_model():
    from huggingface_hub import snapshot_download

    print(f"Downloading {BASE_MODEL}...")

    os.makedirs(MODEL_DIR, exist_ok=True)

    snapshot_download(
        BASE_MODEL,
        local_dir=MODEL_DIR,
        # ignore_patterns=["*.pt"],  # Using safetensors
    )
    # move_cache()

In [ ]:
IMAGE = (
     Image.debian_slim().pip_install_from_requirements("./finetune-requirements.txt")
    .env({"HF_HUB_ENABLE_HF_TRANSFER": "1"})
    .run_function(download_model)
)

In [ ]:
with IMAGE.imports():
    import torch
    import gc
    from sentence_transformers import datasets, SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
    from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
    from sentence_transformers.util import cos_sim
    from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss, DenoisingAutoEncoderLoss
    from sentence_transformers.training_args import BatchSamplers
    import nltk
    from torch.utils.data import DataLoader
    

In [ ]:
@app.cls(
    gpu=GPU, 
    timeout=TIMEOUT,
    scaledown_window=TIMEOUT,
    # allow_concurrent_inputs=NUM_CONCURRENT_REQUESTS,
    image=IMAGE, 
    secrets=SECRETS
)
class ModelMRL:
    model_name_or_path: str = MODEL_DIR
    base_model: str = BASE_MODEL
    device: str = "cuda"

    @enter()
    def load_model(self):

        print(f"Loading {self.model_name_or_path}...")

        self.model = SentenceTransformer(
            self.model_name_or_path, 
            device=self.device, 
            trust_remote_code=True
        )

        self.model_train = SentenceTransformer(
            self.model_name_or_path,
            trust_remote_code=True,
            model_kwargs={"attn_implementation": "sdpa"},
            model_card_data=SentenceTransformerModelCardData(
                language="en",
                license="apache-2.0",
                model_name="Biomedical MRL",
            )
        )

        return self
    
    @exit() 
    def exit(self):
        del self.model
        del self.model_train
        torch.cuda.synchronize()
        gc.collect()


    @method() 
    def load_evaluators(self, corpus, queries, relevant_docs, matryoshka_dimensions: list[int], 
                        query_prompt=None, query_prompt_name=None, corpus_prompt=None, corpus_prompt_name=None):
        
        self.matryoshka_dimensions = matryoshka_dimensions # Important: large to small
      
        matryoshka_evaluators = []

        for dim in self.matryoshka_dimensions:
            print(dim)
            ir_evaluator = InformationRetrievalEvaluator(
                queries=queries,
                corpus=corpus,
                relevant_docs=relevant_docs,
                query_prompt=query_prompt,  
                query_prompt_name=query_prompt_name, 
                corpus_prompt=corpus_prompt, 
                corpus_prompt_name=corpus_prompt_name,
                name=f"dim_{dim}",
                truncate_dim=dim,  
                score_functions={"cosine": cos_sim},
                batch_size=32,
                show_progress_bar=True,
            )
           
            matryoshka_evaluators.append(ir_evaluator)


        self.evaluator = SequentialEvaluator(matryoshka_evaluators)
    
    @method() 
    def evaluate(self):
        results = self.evaluator(self.model)
        
        return results 
    
    @method()
    def evaluate_trained_model(self, model_id):
        fine_tuned_model = SentenceTransformer(
            model_id, device="cuda" if torch.cuda.is_available() else "cpu"
        )

        results = self.evaluator(fine_tuned_model)

        return results

        
    @method() 
    def finetune(self, train_dataset, output_dir, kwargs):
        base_loss = MultipleNegativesRankingLoss(self.model_train)

        train_loss = MatryoshkaLoss(
            self.model_train, base_loss, matryoshka_dims=self.matryoshka_dimensions
        )

        wandb.login(key=os.environ["WANDB_API_KEY"])

        args = SentenceTransformerTrainingArguments(**kwargs)

        trainer = SentenceTransformerTrainer(
            model=self.model_train,
            args=args,
            train_dataset=train_dataset.select_columns(
                ["anchor", "positive"]
            ),  
            loss=train_loss,
            evaluator=self.evaluator,
        )

        trainer.train()

        trainer.model = trainer.accelerator.unwrap_model(trainer.model)
    
        trainer.save_model()

        wandb.finish()

        trainer.model.push_to_hub(output_dir, token=os.environ["HUGGINGFACEHUB_API_TOKEN"])

    @method() 
    def tsdae(self, sentences, output_dir):

        nltk.download('punkt_tab')

        model_tsdae = SentenceTransformer(
            self.model_name_or_path,
            trust_remote_code=True,
            model_card_data=SentenceTransformerModelCardData(
                language="en",
                license="apache-2.0",
                model_name="Biomedical MRL",
            )
        )

        train_dataset = datasets.DenoisingAutoEncoderDataset(sentences)

        train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

        train_loss = DenoisingAutoEncoderLoss(
            model_tsdae, decoder_name_or_path=self.base_model, tie_encoder_decoder=True
        )

        model_tsdae.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=1,
            weight_decay=0,
            scheduler="constantlr",
            optimizer_params={"lr": 2e-5},
            show_progress_bar=True,
        )

        model_tsdae.push_to_hub(output_dir, token=os.environ["HUGGINGFACEHUB_API_TOKEN"])
        

## Evaluate Base Model

In [ ]:
with app.run():
    model = ModelMRL()  
    model.load_evaluators.remote(
        corpus=corpus, 
        queries=queries, 
        relevant_docs=relevant_docs, 
        matryoshka_dimensions=matryoshka_dimensions, 
        query_prompt_name=query_prompt_name,
    )
    base_results = model.evaluate.remote() 

In [ ]:
print("Base Model Evaluation Results")

df = convert_results_to_df(base_results)
df

In [ ]:
save_dir = "data/finetune/base"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
df.to_csv(f"{save_dir}/{BASE_MODEL}-eval-base.csv")

## Train

In [ ]:
output_dir = "snowflake-embed-mrl-train40k"
run_name="snowflake-embed-mrl-train40k"

In [ ]:
training_kwargs = dict(
    output_dir=output_dir,                                     
    prompts=prompts, 
    num_train_epochs=4,                                        
    per_device_train_batch_size=32,                            
    gradient_accumulation_steps=16,                            
    per_device_eval_batch_size=16,                             
    warmup_ratio=0.1,                                          
    learning_rate=2e-5,                                        
    lr_scheduler_type="cosine",                                
    optim="adamw_torch_fused",                                 
    tf32=True,                                                 
    bf16=True,                                                 
    batch_sampler=BatchSamplers.NO_DUPLICATES,                 
    eval_strategy="epoch",                                     
    save_strategy="epoch",                                     
    logging_steps=10,                                          
    save_total_limit=3,                                        
    load_best_model_at_end=True,                               
    metric_for_best_model="eval_dim_128_cosine_ndcg@10",       
    report_to="wandb",                                         
    run_name=run_name, 
)

In [ ]:
with app.run():
    model = ModelMRL()
    model.load_evaluators.remote(
        corpus=corpus, 
        queries=queries, 
        relevant_docs=relevant_docs, 
        matryoshka_dimensions=matryoshka_dimensions, 
        query_prompt_name=query_prompt_name,
    )
    model.finetune.remote(
        train_dataset=train_dataset,
        output_dir = output_dir, 
        kwargs=training_kwargs
    )

## Evaluate Trained Model

In [ ]:
hf_username = "potsu-potsu"  
ft_model_id = f"{hf_username}/{output_dir}" 
ft_model_id

In [ ]:
with app.run():
    model = ModelMRL()
    model.load_evaluators.remote(
        corpus=corpus, 
        queries=queries, 
        relevant_docs=relevant_docs, 
        matryoshka_dimensions=matryoshka_dimensions, 
        query_prompt_name=query_prompt_name,
    )
    ft_results = model.evaluate_trained_model.remote(ft_model_id) 

In [ ]:
print("Fine Tuned Model Evaluation Results")

df = convert_results_to_df(ft_results)
df

In [ ]:
save_dir = "data/finetune/finetuned"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
df.to_csv(f"{save_dir}/{BASE_MODEL}-eval-finetuned.csv")